# Phase 1: Hybrid Legal Language Model

This model combines causal language modelling and masked next-token prediction.
Both objectives use the same transformer. I alternate between the two objectives
while training on the legal corpus.

The model has 12 layers, 8 attention heads, a hidden size of 512 and gated
feed-forward layers. It uses learned positional embeddings and a context of
512 tokens. PyTorch SDPA is used to compute attention.

Reference: *GPT or BERT: why not both?* https://arxiv.org/abs/2410.24159


## 1. Check GPU

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

## 2. Configuration
If batch size 8 does not fit in GPU memory, use batch size 4 and accumulation 4 before starting.


In [ ]:
import os
import random
import json
import time
import math
import zipfile
import shutil

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_FILE = ("Phase_1_Pretraining/Train/train_acts.json",
              "Phase_1_Pretraining/Train/train_high_courts.json",
              "Phase_1_Pretraining/Train/train_supreme_court.json",
              "Phase_1_Pretraining/Train/train_tribunals.json")
TEST_FILE = "Phase_1_Pretraining/Test/test.json"
SEED = 42
random.seed(SEED)

TRAIN_TOKEN_BUDGET = 400_000_000
VAL_TOKENS = 500_000
TRAIN_BLOCK_SIZE = 512
MODEL_CONTEXT_LENGTH = TRAIN_BLOCK_SIZE
N_EMBD = 512
N_LAYER = 12
N_HEAD = 8
FFN_DIM = 1664
TRAIN_BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 4
LEARNING_RATE = 3e-4
WARMUP_UPDATES = 1000
LOG_EVERY = 100
EVAL_EVERY = 1000
OBJECTIVE_PATTERN = ('causal', 'masked')
MASK_START = 0.30
MASK_END = 0.15

TARGET_GENERATION_WORDS = 128
MAX_GENERATION_TOKENS = 8192
GENERATION_MODE = 'top_k'
TEMPERATURE = 0.8
TOP_K = 50
REPETITION_PENALTY = 1.15

OUTPUT_DIR = '/kaggle/working/hybrid-phase1-simple-400m'
MODEL_OUTPUT_DIR = os.path.join(OUTPUT_DIR, 'legal-gpt-bert-final')
PREDICTION_PATH = os.path.join(OUTPUT_DIR, 'prediction.json')
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, 'submission.zip')
LOSS_PLOT_PATH = os.path.join(MODEL_OUTPUT_DIR, 'training_validation_loss.png')
os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)
print('Configuration loaded.')


## 3. Hugging Face login


In [ ]:
from huggingface_hub import login, hf_hub_download
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
login(token=HF_TOKEN)


## 4. Load Tokenizer
Reuse the byte-level BPE implementation. The first four special tokens stay in their original order; `<mask>` is added as a fifth special token for corruption. The tokenizer is trained only on the training-document split. Do not reuse checkpoints with a different vocabulary mapping.


In [ ]:
from collections import defaultdict, Counter
import heapq
SPECIALS = ['<pad>', '<unk>', '<s>', '</s>', '<mask>']

class BytelevelBPE:

    def __init__(self, vocab_size=5000):
        self.vocab_size = vocab_size
        self.specials = SPECIALS.copy()
        self.byte_vocab = [str(i) for i in range(256)]
        self.merges = []
        self.merge_ranks = {}
        self.vocab = []
        self.stoi = {}
        self.itos = {}
        self.symbol_to_bytes = {}

    def word_to_symbols(self, word):
        byte_seq = word.encode('utf-8')
        return [str(b) for b in byte_seq] + ['</w>']

    def build_word_freqs(self, corpus):
        word_counter = Counter()
        print('Building word frequencies...')
        for text in corpus:
            if not isinstance(text, str):
                continue
            word_counter.update(text.split())
        print(f'Unique whitespace-separated words: {len(word_counter):,}')
        word_freqs = {}
        for word, freq in word_counter.items():
            symbols = tuple(self.word_to_symbols(word))
            word_freqs[symbols] = word_freqs.get(symbols, 0) + freq
        return word_freqs

    @staticmethod
    def get_pairs(symbols):
        return {(symbols[i], symbols[i + 1]) for i in range(len(symbols) - 1)}

    @staticmethod
    def pair_counts_in_word(symbols):
        counts = Counter()
        for i in range(len(symbols) - 1):
            counts[symbols[i], symbols[i + 1]] += 1
        return counts

    @staticmethod
    def merge_symbols(symbols, pair, merged_symbol):
        new_symbols = []
        i = 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == pair[0] and (symbols[i + 1] == pair[1]):
                new_symbols.append(merged_symbol)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        return tuple(new_symbols)

    def train(self, corpus):
        word_freqs = self.build_word_freqs(corpus)
        vocab = self.specials + self.byte_vocab + ['</w>']
        seen = set(vocab)
        for b in range(256):
            self.symbol_to_bytes[str(b)] = bytes([b])
        self.symbol_to_bytes['</w>'] = b''
        words = []
        frequencies = []
        for symbols, freq in word_freqs.items():
            words.append(symbols)
            frequencies.append(freq)
        del word_freqs
        pair_counts = defaultdict(int)
        pair_to_words = defaultdict(set)
        print('Building initial pair statistics...')
        for word_id, symbols in enumerate(words):
            freq = frequencies[word_id]
            local_counts = self.pair_counts_in_word(symbols)
            for pair, count in local_counts.items():
                pair_counts[pair] += count * freq
                pair_to_words[pair].add(word_id)
        print(f'Initial unique pairs: {len(pair_counts):,}')
        heap = []
        for pair, count in pair_counts.items():
            heapq.heappush(heap, (-count, pair))
        target_merges = max(0, self.vocab_size - len(vocab))
        print(f'Target merges: {target_merges:,}')
        while len(vocab) < self.vocab_size and heap:
            while heap:
                neg_count, pair = heapq.heappop(heap)
                heap_count = -neg_count
                current_count = pair_counts.get(pair, 0)
                if current_count == heap_count and current_count > 0:
                    break
            else:
                break
            max_freq = pair_counts.get(pair, 0)
            if max_freq <= 1:
                break
            merged_symbol = f'<bpe_{len(self.merges)}>'
            left_bytes = self.symbol_to_bytes[pair[0]]
            right_bytes = self.symbol_to_bytes[pair[1]]
            self.symbol_to_bytes[merged_symbol] = left_bytes + right_bytes
            self.merges.append(pair)
            self.merge_ranks[pair] = (len(self.merges) - 1, merged_symbol)
            if merged_symbol not in seen:
                seen.add(merged_symbol)
                vocab.append(merged_symbol)
            affected_word_ids = list(pair_to_words.get(pair, set()))
            changed_pairs = set()
            for word_id in affected_word_ids:
                old_symbols = words[word_id]
                freq = frequencies[word_id]
                if pair not in self.get_pairs(old_symbols):
                    continue
                old_counts = self.pair_counts_in_word(old_symbols)
                new_symbols = self.merge_symbols(old_symbols, pair, merged_symbol)
                if new_symbols == old_symbols:
                    continue
                new_counts = self.pair_counts_in_word(new_symbols)
                words[word_id] = new_symbols
                all_pairs = set(old_counts) | set(new_counts)
                for p in all_pairs:
                    old_n = old_counts.get(p, 0)
                    new_n = new_counts.get(p, 0)
                    delta = (new_n - old_n) * freq
                    if delta != 0:
                        pair_counts[p] += delta
                        changed_pairs.add(p)
                    if new_n > 0:
                        pair_to_words[p].add(word_id)
                    else:
                        pair_to_words[p].discard(word_id)
            pair_to_words[pair].clear()
            for p in changed_pairs:
                count = pair_counts.get(p, 0)
                if count > 0:
                    heapq.heappush(heap, (-count, p))
        self.vocab = vocab
        self.vocab_size = len(vocab)
        self.stoi = {symbol: i for i, symbol in enumerate(vocab)}
        self.itos = {i: symbol for symbol, i in self.stoi.items()}
        print('\nTokenizer training complete.')
        print('Final vocabulary size:', self.vocab_size)
        print('Number of merges:', len(self.merges))

    def apply_bpe(self, word):
        symbols = self.word_to_symbols(word)
        while len(symbols) > 1:
            best_pair = None
            best_rank = float('inf')
            best_merged_symbol = None
            for i in range(len(symbols) - 1):
                pair = (symbols[i], symbols[i + 1])
                merge_info = self.merge_ranks.get(pair)
                if merge_info is not None:
                    rank, merged_symbol = merge_info
                    if rank < best_rank:
                        best_rank = rank
                        best_pair = pair
                        best_merged_symbol = merged_symbol
            if best_pair is None:
                break
            symbols = list(self.merge_symbols(tuple(symbols), best_pair, best_merged_symbol))
        return symbols

    def encode(self, text, add_bos=False, add_eos=False):
        ids = []
        if add_bos:
            ids.append(self.stoi['<s>'])
        for word in text.split():
            symbols = self.apply_bpe(word)
            for symbol in symbols:
                ids.append(self.stoi.get(symbol, self.stoi['<unk>']))
        if add_eos:
            ids.append(self.stoi['</s>'])
        return ids

    def decode(self, ids):
        words = []
        current_bytes = bytearray()
        for token_id in ids:
            symbol = self.itos.get(token_id)
            if symbol is None:
                continue
            if symbol in self.specials:
                continue
            ends_word = self._symbol_ends_word(symbol)
            byte_value = self.symbol_to_bytes.get(symbol, b'')
            current_bytes.extend(byte_value)
            if ends_word:
                try:
                    word = bytes(current_bytes).decode('utf-8')
                    words.append(word)
                except UnicodeDecodeError:
                    words.append('<unk>')
                current_bytes = bytearray()
        if current_bytes:
            try:
                words.append(bytes(current_bytes).decode('utf-8'))
            except UnicodeDecodeError:
                words.append('<unk>')
        return ' '.join(words)

    def _symbol_ends_word(self, symbol):
        if symbol == '</w>':
            return True
        for pair, (_, merged) in self.merge_ranks.items():
            if merged == symbol:
                return self._symbol_ends_word(pair[1])
        return False

    def get_vocab_size(self):
        return len(self.vocab)


## 5. Download Training Data From Hugging Face

In [ ]:
from huggingface_hub import hf_hub_download

train_file_path_acts = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TRAIN_FILE[0],
    repo_type="dataset",
    token=HF_TOKEN
)

train_file_path_high_courts = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TRAIN_FILE[1],
    repo_type="dataset",
    token=HF_TOKEN
)

train_file_path_supreme_court = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TRAIN_FILE[2],
    repo_type="dataset",
    token=HF_TOKEN
)

train_file_path_tribunals = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TRAIN_FILE[3],
    repo_type="dataset",
    token=HF_TOKEN
)

print("Training file:")
print(train_file_path_acts)

print("Training file:")
print(train_file_path_high_courts)

print("Training file:")
print(train_file_path_supreme_court)

print("Training file:")
print(train_file_path_tribunals)


In [ ]:
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

In [ ]:
source_paths = {
    "acts": train_file_path_acts,
    "high_courts": train_file_path_high_courts,
    "supreme_court": train_file_path_supreme_court,
    "tribunals": train_file_path_tribunals,
}


In [ ]:
import os
import json
import random
import hashlib
import numpy as np

SOURCE_RATIOS = {
    "acts": 1,
    "high_courts": 3,
    "supreme_court": 2,
    "tribunals": 2,
}

TOKENIZER_WORD_BUDGET = 4_000_000
VAL_TOKEN_BUDGET = VAL_TOKENS  # 500,000 in your configuration

# Use your chosen document-level validation fraction.
VALIDATION_FRACTION = 0.05

def allocate_budget(total):
    ratio_sum = sum(SOURCE_RATIOS.values())

    budgets = {
        name: total * ratio // ratio_sum
        for name, ratio in SOURCE_RATIOS.items()
    }

    # Handle rounding for budgets not divisible by five.
    remainder = total - sum(budgets.values())
    for name in list(budgets)[:remainder]:
        budgets[name] += 1

    return budgets

word_budgets = allocate_budget(TOKENIZER_WORD_BUDGET)
train_budgets = allocate_budget(TRAIN_TOKEN_BUDGET)
val_budgets = allocate_budget(VAL_TOKEN_BUDGET)

print("Tokenizer words:", word_budgets)
print("Training tokens:", train_budgets)
print("Validation tokens:", val_budgets)


In [ ]:
%pip install -q ijson


In [ ]:
import os
import json
import hashlib
import sqlite3
import ijson

SPLIT_DIR = os.path.join(OUTPUT_DIR, "document_splits")
os.makedirs(SPLIT_DIR, exist_ok=True)

DEDUP_PATH = os.path.join(SPLIT_DIR, "dedup.sqlite")

# Rebuild from scratch when this cell is rerun.
if os.path.exists(DEDUP_PATH):
    os.remove(DEDUP_PATH)

connection = sqlite3.connect(DEDUP_PATH)
connection.execute(
    "CREATE TABLE seen (digest BLOB PRIMARY KEY) WITHOUT ROWID"
)


class JsonlTexts:
    """Reads texts from disk; each iteration starts from the beginning."""

    def __init__(self, path):
        self.path = path

    def __iter__(self):
        with open(self.path, "r", encoding="utf-8") as f:
            for line in f:
                yield json.loads(line)


train_sources = {}
val_sources = {}
split_counts = {}

try:
    for name, source_path in source_paths.items():
        train_path = os.path.join(SPLIT_DIR, f"{name}_train.jsonl")
        val_path = os.path.join(SPLIT_DIR, f"{name}_val.jsonl")

        counts = {
            "train": 0,
            "validation": 0,
            "invalid": 0,
            "duplicates": 0,
        }

        print(f"\nStreaming {name}...", flush=True)

        with (
            open(source_path, "rb") as source,
            open(train_path, "w", encoding="utf-8") as train_out,
            open(val_path, "w", encoding="utf-8") as val_out,
        ):
            # Verify the top-level structure without loading the file.
            first = source.read(1)
            while first and first.isspace():
                first = source.read(1)

            if first != b"[":
                raise ValueError(f"{name}: expected a top-level JSON list.")

            source.seek(0)

            for index, record in enumerate(
                ijson.items(source, "item"), start=1
            ):
                text = (
                    record.get("text")
                    if isinstance(record, dict)
                    else None
                )

                if not isinstance(text, str) or not text.strip():
                    counts["invalid"] += 1
                    continue

                text = " ".join(text.split())
                digest = hashlib.sha256(text.encode("utf-8")).digest()

                cursor = connection.execute(
                    "INSERT OR IGNORE INTO seen VALUES (?)",
                    (digest,),
                )

                if cursor.rowcount == 0:
                    counts["duplicates"] += 1
                    continue

                # Deterministic document-level split.
                split_digest = hashlib.sha256(
                    str(SEED).encode("ascii") + digest
                ).digest()

                fraction = (
                    int.from_bytes(split_digest[:8], "big") / (1 << 64)
                )

                if fraction < VALIDATION_FRACTION:
                    destination = val_out
                    counts["validation"] += 1
                else:
                    destination = train_out
                    counts["train"] += 1

                destination.write(
                    json.dumps(text, ensure_ascii=False) + "\n"
                )

                if index % 10_000 == 0:
                    connection.commit()

        connection.commit()

        train_sources[name] = JsonlTexts(train_path)
        val_sources[name] = JsonlTexts(val_path)
        split_counts[name] = counts

        print(name, counts, flush=True)

finally:
    connection.close()


In [ ]:
def select_word_budget(documents, target_words, source_name):
    selected = []
    collected = 0

    for text in documents:
        remaining = target_words - collected

        if remaining == 0:
            break

        words = text.split()

        if len(words) <= remaining:
            selected.append(text)
            collected += len(words)
        else:
            selected.append(" ".join(words[:remaining]))
            collected += remaining

    if collected != target_words:
        raise ValueError(
            f"{source_name}: needs {target_words:,} words, "
            f"but only {collected:,} are available in its training split."
        )

    return selected

tokenizer_corpus = []

for name, budget in word_budgets.items():
    selected = select_word_budget(
        train_sources[name], budget, name
    )
    tokenizer_corpus.extend(selected)

    print(f"{name}: {budget:,} words")
    del selected

random.Random(SEED).shuffle(tokenizer_corpus)

print("Tokenizer corpus documents:", len(tokenizer_corpus))


In [ ]:
BPE_VOCAB_SIZE = 20000

# Always create a fresh instance when retraining.
tokenizer = BytelevelBPE(vocab_size=BPE_VOCAB_SIZE)
tokenizer.train(tokenizer_corpus)


examples = [
    "Section 12(1): The Court dismissed the appeal.",
    "भारत का सर्वोच्च न्यायालय",
    "₹500 — café",
    "Multiple   spaces\nand\ttabs",
]

for text in examples:
    ids = tokenizer.encode(text)
    print(repr(text), "->", repr(tokenizer.decode(ids)))


print("Tokenizer prepared.")
print("Actual vocabulary:", tokenizer.get_vocab_size())

del tokenizer_corpus
PAD_ID = tokenizer.stoi['<pad>']
MASK_ID = tokenizer.stoi['<mask>']
SPECIAL_IDS = [tokenizer.stoi[s] for s in tokenizer.specials]


## 6. Create Train and Validation Tokens Set

In [ ]:
def write_token_mixture(sources, budgets, output_path):
    total_tokens = sum(budgets.values())

    token_array = np.memmap(
        output_path,
        dtype=np.uint32,
        mode="w+",
        shape=(total_tokens,),
    )

    eos_id = tokenizer.stoi["</s>"]
    vocab_size = tokenizer.get_vocab_size()

    position = 0
    source_ranges = {}

    try:
        for name, target in budgets.items():
            start = position
            written = 0
            documents_used = 0

            print(f"\nEncoding {name}: target {target:,} tokens")

            for text in sources[name]:
                remaining = target - written

                if remaining == 0:
                    break

                ids = tokenizer.encode(text)
                ids.append(eos_id)

                # The final document may be truncated at the token budget.
                chunk = ids[:remaining]

                if not all(0 <= token_id < vocab_size for token_id in chunk):
                    raise ValueError(f"{name}: invalid token ID.")

                count = len(chunk)
                token_array[position:position + count] = np.asarray(
                    chunk, dtype=np.uint32
                )

                position += count
                written += count
                documents_used += 1

            if written != target:
                raise ValueError(
                    f"{name}: needs {target:,} tokens, "
                    f"but only {written:,} are available. "
                    "No documents were repeated."
                )

            source_ranges[name] = {
                "start": start,
                "end": position,
                "tokens": written,
                "documents_used": documents_used,
            }

            print(f"Finished {name}: {written:,} tokens")

        token_array.flush()

    except Exception:
        del token_array
        if os.path.exists(output_path):
            os.remove(output_path)
        raise

    del token_array
    return source_ranges


TRAIN_IDS_PATH = os.path.join(OUTPUT_DIR, "train_ids.uint32")
VAL_IDS_PATH = os.path.join(OUTPUT_DIR, "val_ids.uint32")

train_ranges = write_token_mixture(
    train_sources, train_budgets, TRAIN_IDS_PATH
)

val_ranges = write_token_mixture(
    val_sources, val_budgets, VAL_IDS_PATH
)


In [ ]:
train_ids = np.memmap(
    TRAIN_IDS_PATH, dtype=np.uint32, mode="r"
)

val_ids = np.memmap(
    VAL_IDS_PATH, dtype=np.uint32, mode="r"
)


metadata = {
    "seed": SEED,
    "source_ratios": SOURCE_RATIOS,
    "validation_document_fraction": VALIDATION_FRACTION,
    "tokenizer_training_words": TOKENIZER_WORD_BUDGET,
    "vocab_size": tokenizer.get_vocab_size(),
    "dtype": "uint32",
    "train": train_ranges,
    "validation": val_ranges,
}

with open(
    os.path.join(OUTPUT_DIR, "data_metadata.json"),
    "w",
    encoding="utf-8",
) as f:
    json.dump(metadata, f, indent=2)

print("Training tokens:", f"{len(train_ids):,}")
print("Validation tokens:", f"{len(val_ids):,}")
print("First IDs:", train_ids[:20])

In [ ]:
import os
import json

os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)

tokenizer_state = {
    "vocab_size": tokenizer.get_vocab_size(),
    "specials": tokenizer.specials,
    "vocab": tokenizer.vocab,
    "merges": [list(pair) for pair in tokenizer.merges],
    "merge_ranks": [
        {
            "pair": list(pair),
            "rank": rank,
            "merged_symbol": merged_symbol,
        }
        for pair, (rank, merged_symbol)
        in tokenizer.merge_ranks.items()
    ],
    "symbol_to_bytes": {
        symbol: list(value)
        for symbol, value in tokenizer.symbol_to_bytes.items()
    },
}

tokenizer_path = os.path.join(MODEL_OUTPUT_DIR, "tokenizer.json")
temporary_path = tokenizer_path + ".tmp"

with open(temporary_path, "w", encoding="utf-8") as f:
    json.dump(tokenizer_state, f, ensure_ascii=False)

os.replace(temporary_path, tokenizer_path)

print("Tokenizer saved:", tokenizer_path)


## 7. Token blocks
The token stream is divided into 512-token blocks. EOS separates documents.
The training function creates the causal or masked next-token targets.
Training and validation documents were separated before tokenization.


In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader


class TokenDataset(Dataset):
    def __init__(self, path, seq_len):
        self.path = path
        self.seq_len = seq_len
        self.ids = np.memmap(path, dtype=np.uint32, mode="r")

    def __len__(self):
        return len(self.ids) // self.seq_len

    def __getitem__(self, idx):
        start = idx * self.seq_len

        # Copy only this block and convert to torch.long.
        input_ids = torch.from_numpy(
            np.array(
                self.ids[start:start + self.seq_len],
                dtype=np.int64,
                copy=True,
            )
        )

        return {
            "input_ids": input_ids,
        }


## 8. Create Datasets

In [ ]:
train_dataset = TokenDataset(TRAIN_IDS_PATH, TRAIN_BLOCK_SIZE)
val_dataset = TokenDataset(VAL_IDS_PATH, TRAIN_BLOCK_SIZE)

In [ ]:
# Batch size 1 is only for checking the data pipeline.
check_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=0,
)

batch = next(iter(check_loader))

print("Training blocks:", len(train_dataset))
print("Validation blocks:", len(val_dataset))
print("Input shape:", batch["input_ids"].shape)
print("Dtype:", batch["input_ids"].dtype)


## 9. Define The Model
Each block has gated multi-head attention and a GEGLU feed-forward layer, with residual connections. The same weights process both objectives. `causal=False` gives bidirectional encoder attention. `causal=True` gives left-to-right attention. KV caching is allowed only in causal evaluation, because adding a token in bidirectional mode would change earlier representations.


In [ ]:
import math
from dataclasses import dataclass, asdict
import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class HybridConfig:
    vocab_size: int
    pad_id: int
    mask_id: int
    d_model: int = 512
    n_layers: int = 12
    n_heads: int = 8
    d_ff: int = 1664
    max_seq_len: int = 512
    dropout: float = 0.1


class GatedAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_heads = cfg.n_heads
        self.head_dim = cfg.d_model // cfg.n_heads
        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model, bias=False)
        self.gate = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.out = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.output_norm = nn.LayerNorm(cfg.d_model, elementwise_affine=False)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x, key_valid, causal=True, past=None, use_cache=False):
        b, t, d = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q = q.view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
        past_len = 0 if past is None else past[0].size(2)
        if past is not None:
            k = torch.cat((past[0], k), dim=2)
            v = torch.cat((past[1], v), dim=2)

        allowed = key_valid[:, None, None, :]
        if causal:
            query_positions = torch.arange(past_len, past_len + t, device=x.device)
            key_positions = torch.arange(k.size(2), device=x.device)
            allowed = allowed & (key_positions[None, :] <= query_positions[:, None])[None, None]

        # A fully padded query row gets a harmless fallback key to avoid NaNs.
        first_key = torch.arange(k.size(2), device=x.device).eq(0)
        allowed = allowed | (~allowed.any(dim=-1, keepdim=True) & first_key)
        # Explicit mask also handles offset queries when a KV cache is present.
        context = F.scaled_dot_product_attention(
            q, k, v, attn_mask=allowed,
            dropout_p=self.dropout.p if self.training else 0.0,
            is_causal=False,
        )
        context = context.transpose(1, 2).contiguous().view(b, t, d)
        # Paper Section 2.2: GEGLU gate and non-affine output LayerNorm.
        context = context * F.gelu(self.gate(x))
        output = self.out(self.output_norm(context))
        return output, (k, v) if use_cache else None


In [ ]:
class GatedFeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.value = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.gate = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.output_norm = nn.LayerNorm(cfg.d_ff, elementwise_affine=False)
        self.out = nn.Linear(cfg.d_ff, cfg.d_model, bias=False)

    def forward(self, x):
        hidden = self.value(x) * F.gelu(self.gate(x))
        return self.out(self.output_norm(hidden))


class HybridBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = nn.LayerNorm(cfg.d_model, elementwise_affine=False)
        self.norm2 = nn.LayerNorm(cfg.d_model, elementwise_affine=False)
        self.attn = GatedAttention(cfg)
        self.ffn = GatedFeedForward(cfg)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x, valid, causal, past=None, use_cache=False):
        attention, kv = self.attn(self.norm1(x), valid, causal, past, use_cache)
        x = x + self.dropout(attention)
        x = x + self.dropout(self.ffn(self.norm2(x)))
        return x, kv


class GPTBertLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model, padding_idx=cfg.pad_id)
        self.pos_emb = nn.Embedding(cfg.max_seq_len, cfg.d_model)
        self.embedding_norm = nn.LayerNorm(cfg.d_model, elementwise_affine=False)
        self.dropout = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([HybridBlock(cfg) for _ in range(cfg.n_layers)])
        self.final_norm = nn.LayerNorm(cfg.d_model, elementwise_affine=False)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.apply(self._initialize)
        self.lm_head.weight = self.tok_emb.weight
        # Smaller residual projections keep a deep stack stable at initialization.
        for block in self.blocks:
            nn.init.normal_(block.attn.out.weight, std=0.02 / math.sqrt(2 * cfg.n_layers))
            nn.init.normal_(block.ffn.out.weight, std=0.02 / math.sqrt(2 * cfg.n_layers))
        with torch.no_grad():
            self.tok_emb.weight[cfg.pad_id].zero_()

    def _initialize(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, std=0.02)

    def forward(self, ids, causal=True, cache=None, use_cache=False, last_only=False):
        if (cache is not None or use_cache) and (not causal or self.training):
            raise ValueError('KV caching is only used in causal evaluation mode.')
        offset = 0 if cache is None else cache['length']
        if offset + ids.size(1) > self.cfg.max_seq_len:
            raise ValueError('Context is full: rebuild from the most recent tokens.')
        positions = torch.arange(offset, offset + ids.size(1), device=ids.device)
        x = self.dropout(self.embedding_norm(self.tok_emb(ids) + self.pos_emb(positions)))
        valid = ids.ne(self.cfg.pad_id)
        if cache is not None:
            valid = torch.cat((cache['valid'], valid), dim=1)
        updated = []
        for i, block in enumerate(self.blocks):
            past = None if cache is None else cache['kv'][i]
            x, kv = block(x, valid, causal, past, use_cache)
            if use_cache:
                updated.append(kv)
        logits = self.lm_head(self.final_norm(x[:, -1:] if last_only else x))
        if use_cache:
            return logits, dict(kv=updated, valid=valid, length=offset + ids.size(1))
        return logits


## 10. Training setup

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP_DTYPE = torch.bfloat16 if device.type == 'cuda' and torch.cuda.is_bf16_supported() else torch.float16
torch.manual_seed(SEED)

cfg = HybridConfig(
    vocab_size=tokenizer.get_vocab_size(), pad_id=PAD_ID, mask_id=MASK_ID,
    d_model=N_EMBD, n_layers=N_LAYER, n_heads=N_HEAD, d_ff=FFN_DIM,
    max_seq_len=TRAIN_BLOCK_SIZE,
)
model = GPTBertLM(cfg).to(device)
train_loader = DataLoader(
    train_dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=True,
    num_workers=0, pin_memory=device.type == 'cuda',
)
val_loader = DataLoader(
    val_dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=device.type == 'cuda',
)
# TRAIN_STEPS = len(train_loader)
TRAIN_STEPS = 50000

print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')
print('Microbatch steps:', TRAIN_STEPS)
print('Planned optimizer updates:', math.ceil(TRAIN_STEPS / GRAD_ACCUM_STEPS))
print('Training tokens:', f'{len(train_dataset) * TRAIN_BLOCK_SIZE:,}')
print('Effective batch size:', TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS)


## 11. Training

In [ ]:
def prepare_objective(ids, objective, mask_probability=0.15, generator=None):
    # Both objectives predict original token k+1 from output position k.
    x = ids.clone()
    targets = ids[:, 1:].clone()
    if objective == 'causal':
        targets[targets == PAD_ID] = -100
        return x, targets

    eligible = torch.ones_like(ids, dtype=torch.bool)
    for token_id in SPECIAL_IDS:
        eligible &= ids.ne(token_id)
    eligible[:, 0] = False  # There is no position -1 to predict token 0.
    random_values = torch.rand(ids.shape, generator=generator, device='cpu').to(ids.device)
    selected = (random_values < mask_probability) & eligible
    # Ensure a short, ordinary text block still has at least one masked target.
    for row in range(ids.size(0)):
        if not selected[row].any() and eligible[row].any():
            choices = random_values[row].masked_fill(~eligible[row], 2.0)
            selected[row, choices.argmin()] = True
    x[selected] = MASK_ID
    targets[~selected[:, 1:]] = -100
    return x, targets


def objective_loss(logits, targets):
    return F.cross_entropy(
        logits[:, :-1].reshape(-1, logits.size(-1)),
        targets.reshape(-1), ignore_index=-100, reduction='sum',
    )


@torch.no_grad()
def evaluate_validation(model, val_loader, device, objective='causal'):
    was_training = model.training
    model.eval()
    total_loss = total_targets = 0
    # Restart this local generator for reproducible masked validation targets.
    mask_generator = torch.Generator().manual_seed(SEED + 101)
    for batch in val_loader:
        ids = batch['input_ids'].to(device, non_blocking=True)
        x, targets = prepare_objective(ids, objective, 0.15, mask_generator)
        count = targets.ne(-100).sum().item()
        if count == 0:
            continue
        with torch.autocast(device_type=device.type, dtype=AMP_DTYPE, enabled=device.type == 'cuda'):
            logits = model(x, causal=objective == 'causal')
            loss_sum = objective_loss(logits, targets)
        total_loss += loss_sum.item()
        total_targets += count
    model.train(was_training)
    if total_targets == 0:
        raise ValueError('No validation targets were found.')
    average = total_loss / total_targets
    # Only causal NLL is reported as autoregressive perplexity.
    ppl = math.exp(min(average, 700)) if objective == 'causal' else None
    return average, ppl


In [ ]:
architecture_source = "import math\nfrom dataclasses import dataclass, asdict\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\n@dataclass\nclass HybridConfig:\n    vocab_size: int\n    pad_id: int\n    mask_id: int\n    d_model: int = 512\n    n_layers: int = 12\n    n_heads: int = 8\n    d_ff: int = 1664\n    max_seq_len: int = 512\n    dropout: float = 0.1\n\n\nclass GatedAttention(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.n_heads = cfg.n_heads\n        self.head_dim = cfg.d_model // cfg.n_heads\n        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model, bias=False)\n        self.gate = nn.Linear(cfg.d_model, cfg.d_model, bias=False)\n        self.out = nn.Linear(cfg.d_model, cfg.d_model, bias=False)\n        self.output_norm = nn.LayerNorm(cfg.d_model, elementwise_affine=False)\n        self.dropout = nn.Dropout(cfg.dropout)\n\n    def forward(self, x, key_valid, causal=True, past=None, use_cache=False):\n        b, t, d = x.shape\n        q, k, v = self.qkv(x).chunk(3, dim=-1)\n        q = q.view(b, t, self.n_heads, self.head_dim).transpose(1, 2)\n        k = k.view(b, t, self.n_heads, self.head_dim).transpose(1, 2)\n        v = v.view(b, t, self.n_heads, self.head_dim).transpose(1, 2)\n        past_len = 0 if past is None else past[0].size(2)\n        if past is not None:\n            k = torch.cat((past[0], k), dim=2)\n            v = torch.cat((past[1], v), dim=2)\n\n        allowed = key_valid[:, None, None, :]\n        if causal:\n            query_positions = torch.arange(past_len, past_len + t, device=x.device)\n            key_positions = torch.arange(k.size(2), device=x.device)\n            allowed = allowed & (key_positions[None, :] <= query_positions[:, None])[None, None]\n\n        # A fully padded query row gets a harmless fallback key to avoid NaNs.\n        first_key = torch.arange(k.size(2), device=x.device).eq(0)\n        allowed = allowed | (~allowed.any(dim=-1, keepdim=True) & first_key)\n        # Explicit mask also handles offset queries when a KV cache is present.\n        context = F.scaled_dot_product_attention(\n            q, k, v, attn_mask=allowed,\n            dropout_p=self.dropout.p if self.training else 0.0,\n            is_causal=False,\n        )\n        context = context.transpose(1, 2).contiguous().view(b, t, d)\n        # Paper Section 2.2: GEGLU gate and non-affine output LayerNorm.\n        context = context * F.gelu(self.gate(x))\n        output = self.out(self.output_norm(context))\n        return output, (k, v) if use_cache else None\n\nclass GatedFeedForward(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.value = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)\n        self.gate = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)\n        self.output_norm = nn.LayerNorm(cfg.d_ff, elementwise_affine=False)\n        self.out = nn.Linear(cfg.d_ff, cfg.d_model, bias=False)\n\n    def forward(self, x):\n        hidden = self.value(x) * F.gelu(self.gate(x))\n        return self.out(self.output_norm(hidden))\n\n\nclass HybridBlock(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.norm1 = nn.LayerNorm(cfg.d_model, elementwise_affine=False)\n        self.norm2 = nn.LayerNorm(cfg.d_model, elementwise_affine=False)\n        self.attn = GatedAttention(cfg)\n        self.ffn = GatedFeedForward(cfg)\n        self.dropout = nn.Dropout(cfg.dropout)\n\n    def forward(self, x, valid, causal, past=None, use_cache=False):\n        attention, kv = self.attn(self.norm1(x), valid, causal, past, use_cache)\n        x = x + self.dropout(attention)\n        x = x + self.dropout(self.ffn(self.norm2(x)))\n        return x, kv\n\n\nclass GPTBertLM(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.cfg = cfg\n        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model, padding_idx=cfg.pad_id)\n        self.pos_emb = nn.Embedding(cfg.max_seq_len, cfg.d_model)\n        self.embedding_norm = nn.LayerNorm(cfg.d_model, elementwise_affine=False)\n        self.dropout = nn.Dropout(cfg.dropout)\n        self.blocks = nn.ModuleList([HybridBlock(cfg) for _ in range(cfg.n_layers)])\n        self.final_norm = nn.LayerNorm(cfg.d_model, elementwise_affine=False)\n        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)\n        self.apply(self._initialize)\n        self.lm_head.weight = self.tok_emb.weight\n        # Smaller residual projections keep a deep stack stable at initialization.\n        for block in self.blocks:\n            nn.init.normal_(block.attn.out.weight, std=0.02 / math.sqrt(2 * cfg.n_layers))\n            nn.init.normal_(block.ffn.out.weight, std=0.02 / math.sqrt(2 * cfg.n_layers))\n        with torch.no_grad():\n            self.tok_emb.weight[cfg.pad_id].zero_()\n\n    def _initialize(self, module):\n        if isinstance(module, (nn.Linear, nn.Embedding)):\n            nn.init.normal_(module.weight, std=0.02)\n\n    def forward(self, ids, causal=True, cache=None, use_cache=False, last_only=False):\n        if (cache is not None or use_cache) and (not causal or self.training):\n            raise ValueError('KV caching is only used in causal evaluation mode.')\n        offset = 0 if cache is None else cache['length']\n        if offset + ids.size(1) > self.cfg.max_seq_len:\n            raise ValueError('Context is full: rebuild from the most recent tokens.')\n        positions = torch.arange(offset, offset + ids.size(1), device=ids.device)\n        x = self.dropout(self.embedding_norm(self.tok_emb(ids) + self.pos_emb(positions)))\n        valid = ids.ne(self.cfg.pad_id)\n        if cache is not None:\n            valid = torch.cat((cache['valid'], valid), dim=1)\n        updated = []\n        for i, block in enumerate(self.blocks):\n            past = None if cache is None else cache['kv'][i]\n            x, kv = block(x, valid, causal, past, use_cache)\n            if use_cache:\n                updated.append(kv)\n        logits = self.lm_head(self.final_norm(x[:, -1:] if last_only else x))\n        if use_cache:\n            return logits, dict(kv=updated, valid=valid, length=offset + ids.size(1))\n        return logits\n"
with open(os.path.join(MODEL_OUTPUT_DIR, 'model_arch.py'), 'w', encoding='utf-8') as f:
    f.write(architecture_source)
print('Saved model_arch.py with the SDPA implementation.')


In [ ]:
def train_model(model, train_loader, val_loader, train_steps, lr,
                accumulation_steps=2, log_every=100, eval_every=1000):
    import pickle

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=lr, betas=(0.9, 0.98), weight_decay=0.01
    )
    scaler = torch.amp.GradScaler(
        'cuda', enabled=device.type == 'cuda' and AMP_DTYPE == torch.float16
    )
    total_updates = math.ceil(train_steps / accumulation_steps)
    warmup = min(WARMUP_UPDATES, max(1, total_updates // 10))
    eval_every = math.ceil(eval_every / accumulation_steps) * accumulation_steps

    loss_history, step_history, validation_history = [], [], []
    best_loss = best_masked_loss = float('inf')
    updates = skipped_updates = tokens_seen = 0
    running_loss = running_tokens = 0
    pending = 0
    batches = iter(train_loader)
    model.train()
    optimizer.zero_grad(set_to_none=True)

    for step in range(1, train_steps + 1):
        # Both objectives use the same shuffled data stream.
        try:
            batch = next(batches)
        except StopIteration:
            batches = iter(train_loader)
            batch = next(batches)

        objective = OBJECTIVE_PATTERN[(step - 1) % len(OBJECTIVE_PATTERN)]
        mask_probability = MASK_START + (MASK_END - MASK_START) * (step - 1) / max(1, train_steps - 1)
        ids = batch['input_ids'].to(device, non_blocking=True)
        x, targets = prepare_objective(ids, objective, mask_probability)
        target_count = targets.ne(-100).sum().item()

        with torch.autocast(device_type=device.type, dtype=AMP_DTYPE,
                            enabled=device.type == 'cuda'):
            logits = model(x, causal=objective == 'causal')
            loss_sum = objective_loss(logits, targets)
            loss = loss_sum / target_count

        if not torch.isfinite(loss):
            raise ValueError(f'Loss became non-finite at step {step}.')
        scaler.scale(loss / accumulation_steps).backward()
        pending += 1
        tokens_seen += ids.numel()
        if objective == 'causal':
            running_loss += loss_sum.detach().item()
            running_tokens += target_count
        del ids, x, targets, logits, loss, loss_sum

        if pending == accumulation_steps or step == train_steps:
            scaler.unscale_(optimizer)
            # The final group can contain fewer microbatches.
            if pending < accumulation_steps:
                for parameter in model.parameters():
                    if parameter.grad is not None:
                        parameter.grad.mul_(accumulation_steps / pending)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            if updates < warmup:
                current_lr = lr * (updates + 1) / warmup
            else:
                progress = (updates - warmup) / max(1, total_updates - warmup - 1)
                current_lr = lr * (0.1 + 0.45 * (1 + math.cos(math.pi * min(progress, 1))))
            for group in optimizer.param_groups:
                group['lr'] = current_lr

            old_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            if scaler.get_scale() >= old_scale:
                updates += 1
            else:
                skipped_updates += 1
            optimizer.zero_grad(set_to_none=True)
            pending = 0

        if step == 1 or step % log_every == 0 or step == train_steps:
            if running_tokens:
                train_loss = running_loss / running_tokens
                loss_history.append(train_loss)
                step_history.append(step)
                print(f'Step {step}/{train_steps} | Causal loss: {train_loss:.4f}', flush=True)
            running_loss = running_tokens = 0

        if step % eval_every == 0 or step == train_steps:
            val_loss, val_ppl = evaluate_validation(model, val_loader, device, 'causal')
            masked_loss, _ = evaluate_validation(model, val_loader, device, 'masked')
            validation_history.append({
                'step': step, 'optimizer_steps': updates,
                'tokens_seen': tokens_seen, 'val_loss': val_loss,
                'val_ppl': val_ppl, 'masked_val_loss': masked_loss,
            })
            print(f'Validation PPL: {val_ppl:.2f} | Masked loss: {masked_loss:.4f}', flush=True)

            weights = {
                'model_config': asdict(model.cfg),
                'model_state_dict': model.state_dict(),
                'step': step, 'val_loss': val_loss, 'masked_val_loss': masked_loss,
            }
            if val_loss < best_loss:
                best_loss = val_loss
                torch.save(weights, os.path.join(MODEL_OUTPUT_DIR, 'best.pt'))
            if masked_loss < best_masked_loss:
                best_masked_loss = masked_loss
                torch.save(weights, os.path.join(MODEL_OUTPUT_DIR, 'best_masked.pt'))

            history = {
                'loss_history': loss_history, 'step_history': step_history,
                'validation_history': validation_history,
            }
            # Keep a training backup, even though this notebook starts from scratch.
            torch.save({
                **weights, 'optimizer_state_dict': optimizer.state_dict(),
                'scaler_state_dict': scaler.state_dict(), 'optimizer_steps': updates,
                'skipped_updates': skipped_updates, 'tokens_seen': tokens_seen,
                'train_steps': train_steps, 'accumulation_steps': accumulation_steps,
                'peak_lr': lr, 'warmup_updates': warmup, 'history': history,
            }, os.path.join(MODEL_OUTPUT_DIR, 'latest.pt'))
            with open(os.path.join(MODEL_OUTPUT_DIR, 'history.json'), 'w') as f:
                json.dump(history, f, indent=2)
            with open(os.path.join(MODEL_OUTPUT_DIR, 'history.pkl'), 'wb') as f:
                pickle.dump(history, f)

    checkpoint = torch.load(os.path.join(MODEL_OUTPUT_DIR, 'best.pt'),
                            map_location='cpu', weights_only=True)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    print('Best causal checkpoint:', checkpoint['step'])
    print('Token presentations:', f'{tokens_seen:,}')
    print('Optimizer updates:', updates, '| AMP-skipped updates:', skipped_updates)
    return loss_history, step_history, validation_history


In [ ]:
loss_history, step_history, validation_history = train_model(
    model, train_loader, val_loader,
    train_steps=TRAIN_STEPS, lr=LEARNING_RATE,
    accumulation_steps=GRAD_ACCUM_STEPS,
    log_every=LOG_EVERY, eval_every=EVAL_EVERY,
)


## 12. Perplexity

In [ ]:
val_loss, val_perplexity = evaluate_validation(model, val_loader, device, 'causal')
masked_loss, _ = evaluate_validation(model, val_loader, device, 'masked')
print(f'Causal validation NLL: {val_loss:.4f}')
print(f'Causal validation perplexity: {val_perplexity:.2f}')
print(f'Masked next-token validation NLL: {masked_loss:.4f}')


## 13. Plot Training and Validation Loss

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(step_history, loss_history, label='Train causal loss')
axes[0].plot([r['step'] for r in validation_history],
             [r['val_loss'] for r in validation_history], label='Validation causal loss')
axes[1].plot([r['step'] for r in validation_history],
             [r['masked_val_loss'] for r in validation_history], label='Validation masked loss')
for ax in axes:
    ax.set_xlabel('Microbatch step')
    ax.set_ylabel('Cross-entropy')
    ax.legend()
    ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(LOSS_PLOT_PATH, dpi=150)
plt.show()


## 14. Save Final Model to Kaggle Output

In [ ]:
torch.save({
    'model_config': asdict(model.cfg),
    'model_state_dict': {name: value.detach().cpu()
                         for name, value in model.state_dict().items()},
}, os.path.join(MODEL_OUTPUT_DIR, 'model.pt'))
print('Saved the best causal model as model.pt')


## 15. Download model outputs

In [ ]:
from IPython.display import FileLink, display
archive_path = shutil.make_archive(
    os.path.join(OUTPUT_DIR, 'hybrid-phase1-model-output'),
    'zip', root_dir=MODEL_OUTPUT_DIR,
)
display(FileLink(archive_path))


## 16. Optional: Phase 1 continuation generation

In [ ]:
test_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TEST_FILE,
    repo_type="dataset",
    token=HF_TOKEN
)

print("Test file:", test_file_path)


## 17. Load Test Prompts

In [ ]:
with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)



print("Test examples:", len(test_data))


## 18. Load Final Model for Generation

In [ ]:
# Free the training model's GPU storage before loading the generation model.
if 'model' in globals():
    model.to('cpu')
if torch.cuda.is_available():
    torch.cuda.empty_cache()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Uses the model saved above. Change MODEL_OUTPUT_DIR to load another checkpoint.
checkpoint = torch.load(os.path.join(MODEL_OUTPUT_DIR, 'model.pt'), map_location='cpu', weights_only=True)
generation_cfg = HybridConfig(**checkpoint['model_config'])
generation_model = GPTBertLM(generation_cfg)
generation_model.load_state_dict(checkpoint['model_state_dict'])
del checkpoint
generation_model = generation_model.to(device).eval()
with open(os.path.join(MODEL_OUTPUT_DIR, 'tokenizer.json'), 'r', encoding='utf-8') as f:
    state = json.load(f)
generation_tokenizer = BytelevelBPE(vocab_size=state['vocab_size'])
generation_tokenizer.specials = state['specials']
generation_tokenizer.vocab = state['vocab']
generation_tokenizer.vocab_size = len(state['vocab'])
generation_tokenizer.stoi = {symbol: i for i, symbol in enumerate(state['vocab'])}
generation_tokenizer.itos = {i: symbol for symbol, i in generation_tokenizer.stoi.items()}
generation_tokenizer.merges = [tuple(pair) for pair in state['merges']]
generation_tokenizer.merge_ranks = {tuple(record['pair']): (record['rank'], record['merged_symbol']) for record in state['merge_ranks']}
generation_tokenizer.symbol_to_bytes = {symbol: bytes(values) for symbol, values in state['symbol_to_bytes'].items()}
print("Model and tokenizer loaded.")


## 19. Generate continuations
Generation uses **causal mode only**, with greedy, temperature or top-k sampling. The default uses top-k sampling and a modest repetition penalty on recent generated tokens. These decoding choices do not reduce training loss.

KV caching reuses earlier causal states until the learned-position context is full. At that point, the complete cache is rebuilt from the most recent context tokens, with positions restarting at zero. This is exact for that cropped input, but rebuilding is slower than incremental decoding. We do not simply remove cached keys, which would leave inconsistent positions and representations.

Only continuation words are returned. Special tokens are suppressed and complete nonempty BPE words are counted. A token budget prevents infinite generation; if the model fails to finish 128 words, the function reports the failure instead of padding with invented filler. Previous successful predictions remain saved.


In [ ]:
@torch.inference_mode()
def generate_continuation(
    prompt, model, tokenizer, device,
    target_words=128,
    mode='top_k',
    temperature=0.8,
    top_k=50,
    max_new_tokens=8192,
    repetition_penalty=1.15,
):
    model.eval()
    ids = tokenizer.encode(prompt)
    if not ids:
        raise ValueError('The prompt contains no tokens.')
    context_limit = min(MODEL_CONTEXT_LENGTH, model.cfg.max_seq_len)
    context_ids = torch.tensor([ids[-context_limit:]], dtype=torch.long, device=device)
    generated = []
    word_count = word_bytes = 0
    boundary_id = tokenizer.stoi["</w>"]
    if not hasattr(tokenizer, 'generation_tables'):
        symbols = [tokenizer.itos[i] for i in range(tokenizer.get_vocab_size())]
        tokenizer.generation_tables = (
            [tokenizer._symbol_ends_word(s) for s in symbols],
            [len(tokenizer.symbol_to_bytes.get(s, b'')) for s in symbols],
            [tokenizer.stoi[s] for s in tokenizer.specials],
        )
    ends_word, byte_lengths, special_ids = tokenizer.generation_tables
    empty_ids = [i for i, count in enumerate(byte_lengths) if count == 0]
    use_amp = device.type == 'cuda'
    amp_dtype = torch.bfloat16 if use_amp and torch.cuda.is_bf16_supported() else torch.float16
    with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
        logits, cache = model(context_ids, causal=True, use_cache=True, last_only=True)
    for step in range(max_new_tokens):
        scores = logits[:, -1].float().clone()
        scores[:, special_ids] = float('-inf')
        if word_bytes == 0:
            scores[:, empty_ids] = float('-inf')
        if generated and repetition_penalty != 1:
            recent_ids = [
                token_id
                for token_id in set(generated[-128:])
                if token_id != boundary_id
            ]

            if recent_ids:
                recent = torch.tensor(
                    recent_ids,
                    dtype=torch.long,
                    device=device,
                )

                values = scores[:, recent]

                scores[:, recent] = torch.where(
                    values < 0,
                    values * repetition_penalty,
                    values / repetition_penalty,
                )
        if mode == 'greedy':
            next_token = scores.argmax(dim=-1, keepdim=True)
        elif mode == 'temperature':
            next_token = torch.multinomial((scores / temperature).softmax(-1), 1)
        else:
            values, indices = torch.topk(scores / temperature, min(top_k, scores.size(-1)), dim=-1)
            next_token = indices.gather(-1, torch.multinomial(values.softmax(-1), 1))
        token_id = next_token.item()
        generated.append(token_id)

        word_bytes += byte_lengths[token_id]

        if ends_word[token_id]:
            if word_bytes > 0:
                word_count += 1

            word_bytes = 0

            if word_count >= target_words:
                words = tokenizer.decode(generated).split()

                if len(words) >= target_words:
                    return " ".join(words[:target_words])
        if step + 1 < max_new_tokens:
            context_ids = torch.cat((context_ids, next_token), dim=1)[:, -context_limit:]
            with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
                if cache['length'] >= context_limit:
                    # Rebuild all layers for the cropped context and reset positions.
                    logits, cache = model(context_ids, causal=True, use_cache=True, last_only=True)
                else:
                    logits, cache = model(next_token, causal=True, cache=cache,
                                          use_cache=True, last_only=True)
    text = tokenizer.decode(generated)
    raise RuntimeError(f'Generated only {len(text.split())} words in {len(generated)} tokens. '
                       f'Preview: {text[:250]!r}. Try a better checkpoint or different decoding settings.')


## 20. Generate Predictions for the Test Set

In [ ]:
# Change decoding settings in Configuration, then rerun from here.
print('Decoding:', GENERATION_MODE, '| temperature:', TEMPERATURE, '| top-k:', TOP_K)


In [ ]:
predictions = {}
start_time = time.time()
torch.manual_seed(SEED)

for example_id, record in test_data.items():
    continuation = generate_continuation(
        prompt=record['text'], model=generation_model,
        tokenizer=generation_tokenizer, device=device,
        target_words=TARGET_GENERATION_WORDS, mode=GENERATION_MODE,
        temperature=TEMPERATURE, top_k=TOP_K,
        max_new_tokens=MAX_GENERATION_TOKENS,
        repetition_penalty=REPETITION_PENALTY,
    )
    predictions[example_id] = {'text': continuation}
    with open(PREDICTION_PATH, 'w', encoding='utf-8') as f:
        json.dump(predictions, f, ensure_ascii=False, indent=2)
    print(f'Completed {len(predictions)}/{len(test_data)} | '
          f'Elapsed: {(time.time() - start_time) / 60:.1f} min', flush=True)


## 21. Save prediction.json

In [ ]:
if set(predictions) != set(test_data) or any(len(r['text'].split()) != TARGET_GENERATION_WORDS for r in predictions.values()):
    raise ValueError('Complete all 128-word predictions before saving the submission.')
import os
import json



PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")

with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

print("Saved:", PREDICTION_PATH)
print("Predictions:", len(predictions))


## 22. Create CodaBench Submission ZIP

In [ ]:
import zipfile

SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")

with zipfile.ZipFile(
    SUBMISSION_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    archive.write(
        PREDICTION_PATH,
        arcname="prediction.json",
    )

print('Submission ZIP saved:', SUBMISSION_ZIP)
